# First-Order Logic (FOL) Syntax and Semantics

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand First-Order Logic (FOL) syntax and semantics
- Work with predicates, quantifiers (∀, ∃), and variables
- Write code to parse and evaluate FOL formulas
- Apply logical reasoning with FOL to solve AI problems
- Understand the difference between propositional logic and FOL

## 🔗 Where this fits

**Builds on:** Course 02 — Unit 2, lesson 03 "Inference Rules and Logical Reasoning" — the same reasoning, extended with predicates, variables and quantifiers so it can talk about *all* and *some*.

**Used later in:** Course 02 — Unit 3, which replaces certainty with probability once the world stops being fully known.

---

This notebook covers practical activities from **Course 02, Unit 2**:
- Working with First-Order Logic (FOL) syntax and semantics
- Writing code to parse and evaluate FOL formulas
- Applying logical reasoning to solve AI problems (like knowledge graph reasoning)

---

## Introduction to First-Order Logic

**First-Order Logic (FOL)** extends propositional logic by adding:
- **Predicates**: Properties of objects (e.g., Human(x), Mortal(x))
- **Quantifiers**: Universal (∀) and Existential (∃)
- **Variables**: Represent objects (x, y, z)
- **Functions**: Operations on objects

**Key Difference from Propositional Logic:**
- Propositional Logic: Works with simple propositions (True/False)
- FOL: Can express relationships and properties of objects


## 🌍 The case: quantifiers you have already used without knowing it

First-order logic sounds like philosophy. You have been running it since your first `SELECT`.

**Marseille, 1972.** Alain Colmerauer and Philippe Roussel built **Prolog**, a language whose programs *are* first-order clauses and whose execution *is* proof search. `grandparent(X, Z) :- parent(X, Y), parent(Y, Z).` is a program, a rule, and a logical formula at the same time — and it is the same rule you will apply by hand in Part 5 below.

**1970, IBM San Jose.** Edgar Codd's relational model defined database queries as **relational calculus** — quantified predicate formulas over tables. That is why SQL reads the way it does: `WHERE EXISTS (SELECT ...)` is ∃, `WHERE NOT EXISTS (SELECT ... WHERE NOT ...)` is how you spell ∀. Every query you have written was a first-order formula being evaluated against a finite domain, exactly like the evaluator in Part 4b.

**Today, on your repository.** **CodeQL** (the query language QL first appeared at Semmle in 2007; GitHub acquired the company in 2019) compiles a codebase into a relational database of program facts — declarations, call edges, data flow — and then hunts security vulnerabilities with a declarative query language whose semantics are **Datalog**, a decidable fragment of first-order logic. "Find every path from user input to a SQL string" is a quantified logical query, and it runs on millions of lines of code.

### What goes wrong without this

Propositional logic, from the last two notebooks, needs **one symbol per fact**. To express "every human is mortal" for a population of eight billion, you would need eight billion separate propositions and eight billion separate rules — and the moment a new person is born, none of your rules apply to them.

That is the whole failure quantifiers fix. `∀x. Human(x) → Mortal(x)` is one sentence that covers every object in the domain, including the ones you have not met yet. Without variables and predicates, a rule cannot generalise; it can only be repeated.

---


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# Import what this notebook needs: `re` powers the small formula parser (Part 3),
# and `typing` hints make the class interfaces easier to read.
import re
from typing import Dict, List, Set, Callable, Any

print("✅ Libraries imported!")
print("Ready to work with First-Order Logic!")

✅ Libraries imported!
Ready to work with First-Order Logic!


## Part 1: FOL Syntax - Predicates and Terms

Let's start by representing FOL predicates and terms in Python.


In [2]:
# A predicate is FOL's basic sentence part: a relation name applied to arguments, e.g. Human(socrates).
# Why: propositional logic can only say 'P is true'; predicates let us talk ABOUT objects.

class FOLPredicate:
    """Represents a FOL predicate (e.g., Human(x), Loves(x, y))"""

    def __init__(self, name: str, arguments: list):
        self.name = name              # Predicate name, e.g. "Human"
        self.arguments = arguments    # List of terms, e.g. ["socrates"]

    def __repr__(self):
        return f"{self.name}({', '.join(self.arguments)})"


# Quick check: build and display a predicate
p1 = FOLPredicate("Human", ["socrates"])
print(f"Predicate: {p1}")
print(f"  name      = {p1.name}")
print(f"  arguments = {p1.arguments}")

Predicate: Human(socrates)
  name      = Human
  arguments = ['socrates']


## Part 2: Quantifiers (Universal ∀ and Existential ∃)

Quantifiers allow us to express statements about all objects or some objects.


In [3]:
# A formula wraps a predicate, optionally under a quantifier: 'for all x' or 'there exists x'.
# Why: quantifiers are the big upgrade over propositional logic - one formula can cover EVERY object.

class FOLFormula:
    """Represents a FOL formula: a bare predicate or a quantified formula."""

    def __init__(self, kind: str, variable: str = None, *, predicate=None, subformula=None):
        self.kind = kind              # 'predicate', 'universal' (∀) or 'existential' (∃)
        self.variable = variable      # Quantified variable name, e.g. 'x'
        self.predicate = predicate    # FOLPredicate when kind == 'predicate'
        self.subformula = subformula  # Inner FOLFormula for quantified formulas

    def __repr__(self):
        if self.kind == "predicate":
            return repr(self.predicate)
        symbol = {"universal": "∀", "existential": "∃"}.get(self.kind, "?")
        return f"{symbol}{self.variable}. {self.subformula}"


# Quick check: ∀x. Human(x)
inner = FOLFormula("predicate", predicate=FOLPredicate("Human", ["x"]))
print(FOLFormula("universal", "x", subformula=inner))

∀x. Human(x)


## Part 3: Simple FOL Formula Parser

Let's create a simple parser to convert FOL formulas from string representation to our data structure.


In [4]:
# Parse text like 'Loves(romeo, juliet)' or '∀x. Human(x)' into the objects above using regular expressions.
# Why: real systems read logic from text files or user input - parsing turns strings into structures we can evaluate.

class FOLParser:
    """Simple parser for FOL formulas"""
    
    @staticmethod
    def parse_predicate(pred_str: str) -> FOLPredicate:
        """Parse a predicate string like 'Human(x)' or 'Loves(x, y)'"""
        # Match pattern: Name(arg1, arg2, ...)
        match = re.match(r'(\w+)\(([^)]+)\)', pred_str.strip())
        if not match:
            raise ValueError(f"Invalid predicate format: {pred_str}")
        
        name = match.group(1)
        args_str = match.group(2)
        arguments = [arg.strip() for arg in args_str.split(',')]
        
        return FOLPredicate(name, arguments)
    
    @staticmethod
    def parse_universal(formula_str: str) -> FOLFormula:
        """Parse universal quantifier: ∀x. P(x)"""
        # Match: ∀x. formula
        match = re.match(r'∀(\w+)\.\s*(.+)', formula_str.strip())
        if not match:
            raise ValueError(f"Invalid universal quantifier: {formula_str}")
        
        variable = match.group(1)
        subformula_str = match.group(2).strip()
        
        # For simplicity, parse the subformula as a predicate
        predicate = FOLParser.parse_predicate(subformula_str)
        subformula = FOLFormula('predicate', predicate=predicate)
        
        return FOLFormula('universal', variable, subformula=subformula)

    @staticmethod
    def parse_existential(formula_str: str) -> FOLFormula:
        """Parse existential quantifier: ∃x. P(x)"""
        # Match: ∃x. formula
        match = re.match(r'∃(\w+)\.\s*(.+)', formula_str.strip())
        if not match:
            raise ValueError(f"Invalid existential quantifier: {formula_str}")

        variable = match.group(1)
        subformula_str = match.group(2).strip()

        # For simplicity, parse the subformula as a predicate
        predicate = FOLParser.parse_predicate(subformula_str)
        subformula = FOLFormula('predicate', predicate=predicate)

        return FOLFormula('existential', variable, subformula=subformula)

# Test parser
print("=" * 60)
print("FOL Formula Parser:")
print("=" * 60)

# Parse predicates
pred1 = FOLParser.parse_predicate("Human(socrates)")
print(f"Parsed: {pred1}")

pred2 = FOLParser.parse_predicate("Loves(romeo, juliet)")
print(f"Parsed: {pred2}")

# Parse universal quantifier
universal = FOLParser.parse_universal("∀x. Human(x)")
print(f"Parsed: {universal}")

# Parse existential quantifier
existential = FOLParser.parse_existential("∃x. Mortal(x)")
print(f"Parsed: {existential}")

FOL Formula Parser:
Parsed: Human(socrates)
Parsed: Loves(romeo, juliet)
Parsed: ∀x. Human(x)
Parsed: ∃x. Mortal(x)


## Part 4: Evaluating FOL Formulas

Now let's create a simple evaluator that can check if FOL formulas are true given a knowledge base.


In [5]:
# A tiny knowledge base: store ground facts and look them up.
# Why: a formula is only true or false RELATIVE to a set of known facts - this is that fact store.
# Note the closed-world assumption: anything not stored is treated as False.

class KnowledgeBase:
    """Simple knowledge base for storing facts"""
    
    def __init__(self):
        self.facts = []  # List of (predicate_name, arguments_tuple) -> bool
    
    def add_fact(self, predicate: FOLPredicate, value: bool = True):
        """Add a fact to the knowledge base"""
        self.facts.append((predicate.name, tuple(predicate.arguments), value))
    
    def check(self, predicate: FOLPredicate) -> bool:
        """Check if a predicate is true in the knowledge base"""
        fact_key = (predicate.name, tuple(predicate.arguments))
        for fact_name, fact_args, fact_value in self.facts:
            if fact_name == predicate.name and fact_args == tuple(predicate.arguments):
                return fact_value
        return False  # Closed world assumption: not found = false
    
    def get_all(self, predicate_name: str, variable_positions: List[int]):
        """Get all facts matching a pattern (for quantifier evaluation)"""
        results = []
        for fact_name, fact_args, fact_value in self.facts:
            if fact_name == predicate_name:
                results.append((fact_args, fact_value))
        return results

# Example: Socrates is mortal argument in FOL
print("=" * 60)
print("FOL Reasoning Example: Socrates is Mortal")
print("=" * 60)

# Create knowledge base
kb = KnowledgeBase()

# Add facts
kb.add_fact(FOLPredicate("Human", ["socrates"]), True)
kb.add_fact(FOLPredicate("Human", ["plato"]), True)

# Add rule: All humans are mortal
# This means: For any x, if Human(x) is true, then Mortal(x) is true

# Check if Socrates is human
socrates_human = FOLPredicate("Human", ["socrates"])
if kb.check(socrates_human):
    print("✅ Socrates is Human (fact in KB)")
    
    # Since all humans are mortal (rule), and Socrates is human, then:
    # Socrates is mortal
    print("✅ All humans are mortal (rule)")
    print("✅ Therefore: Socrates is mortal (inferred)")
    
    # Add the inferred fact
    kb.add_fact(FOLPredicate("Mortal", ["socrates"]), True)
    print(f"✅ Added to KB: Mortal(socrates)")

FOL Reasoning Example: Socrates is Mortal
✅ Socrates is Human (fact in KB)
✅ All humans are mortal (rule)
✅ Therefore: Socrates is mortal (inferred)
✅ Added to KB: Mortal(socrates)


### Part 4b: Evaluating Quantified Formulas (∀ and ∃)

So far we only checked *ground* facts. Now let's actually **evaluate quantified formulas** against the knowledge base:

- **∀x. P(x)** is true iff P holds for **every** object in the domain
- **∃x. P(x)** is true iff P holds for **at least one** object in the domain

Our evaluator handles the simplest formula shape (one quantifier over one predicate) — enough to see exactly how quantifier semantics work over a finite domain.

In [6]:
# Evaluate quantified formulas: substitute every known object for the variable, then combine with all()/any().
# Why: this shows exactly what ∀ ('all objects pass') and ∃ ('at least one passes') mean in code.

def domain_of(kb: KnowledgeBase) -> list:
    """The finite domain of discourse: every constant mentioned in the KB's facts."""
    objects = []
    for fact_name, fact_args, fact_value in kb.facts:
        for arg in fact_args:
            if arg not in objects:
                objects.append(arg)
    return objects

def evaluate_formula(kb: KnowledgeBase, formula: FOLFormula, domain: list) -> bool:
    """Evaluate a FOL formula (ground predicate or quantified predicate) against a KB."""
    if formula.kind == 'predicate':
        return kb.check(formula.predicate)

    if formula.kind in ('universal', 'existential'):
        pred = formula.subformula.predicate
        # Substitute each domain object for the quantified variable and check
        results = []
        for obj in domain:
            args = [obj if a == formula.variable else a for a in pred.arguments]
            results.append(kb.check(FOLPredicate(pred.name, args)))
        return all(results) if formula.kind == 'universal' else any(results)

    raise ValueError(f"Unknown formula kind: {formula.kind}")

# Evaluate quantified formulas against the KB from Part 4
# (KB contains: Human(socrates), Human(plato), Mortal(socrates))
domain = domain_of(kb)
print("=" * 60)
print("Evaluating Quantified Formulas Against the KB")
print("=" * 60)
print(f"Domain of discourse: {domain}")
print()

formulas = [
    FOLParser.parse_universal("∀x. Human(x)"),
    FOLParser.parse_universal("∀x. Mortal(x)"),
    FOLParser.parse_existential("∃x. Mortal(x)"),
    FOLParser.parse_existential("∃x. Robot(x)"),
]

for formula in formulas:
    value = evaluate_formula(kb, formula, domain)
    print(f"  {str(formula):<18} -> {value}")

print()
print("Reading the results:")
print("  ∀x. Human(x) is True  — every object in the domain is asserted Human")
print("  ∀x. Mortal(x) is False — Mortal(plato) is not in the KB (closed world)")
print("  ∃x. Mortal(x) is True  — Mortal(socrates) was added in Part 4")
print("  ∃x. Robot(x)  is False — no Robot facts exist")

Evaluating Quantified Formulas Against the KB
Domain of discourse: ['socrates', 'plato']

  ∀x. Human(x)       -> True
  ∀x. Mortal(x)      -> False
  ∃x. Mortal(x)      -> True
  ∃x. Robot(x)       -> False

Reading the results:
  ∀x. Human(x) is True  — every object in the domain is asserted Human
  ∀x. Mortal(x) is False — Mortal(plato) is not in the KB (closed world)
  ∃x. Mortal(x) is True  — Mortal(socrates) was added in Part 4
  ∃x. Robot(x)  is False — no Robot facts exist


## Part 5: Applying FOL to AI Reasoning Problems

Let's apply FOL to a practical AI reasoning problem.

*Note*: the rule below (`Parent(x, y) ∧ Parent(y, z) → Grandparent(x, z)`) is applied **manually** — the code checks the two ground facts and then adds the conclusion. A general rule engine that applies such rules automatically is what forward chaining does in `01_knowledge_representation.ipynb` (Part 4).


In [7]:
# Example: Knowledge Graph Reasoning with FOL
# We'll represent relationships and infer new knowledge

print("=" * 60)
print("AI Reasoning Problem: Family Relationships")
print("=" * 60)

family_kb = KnowledgeBase()

# Add facts about family relationships
family_kb.add_fact(FOLPredicate("Parent", ["alice", "bob"]), True)
family_kb.add_fact(FOLPredicate("Parent", ["bob", "charlie"]), True)

# Rule: Grandparent relation
# ∀x, y, z. Parent(x, y) ∧ Parent(y, z) → Grandparent(x, z)

# Check: Is Alice a grandparent of Charlie?
alice_parent_bob = FOLPredicate("Parent", ["alice", "bob"])
bob_parent_charlie = FOLPredicate("Parent", ["bob", "charlie"])

if family_kb.check(alice_parent_bob) and family_kb.check(bob_parent_charlie):
    print("✅ Parent(alice, bob) is True")
    print("✅ Parent(bob, charlie) is True")
    print("✅ Applying rule: Parent(x, y) ∧ Parent(y, z) → Grandparent(x, z)")
    print("✅ Therefore: Grandparent(alice, charlie)")
    
    # Add inferred fact
    family_kb.add_fact(FOLPredicate("Grandparent", ["alice", "charlie"]), True)
    print("✅ Added to KB: Grandparent(alice, charlie)")


AI Reasoning Problem: Family Relationships
✅ Parent(alice, bob) is True
✅ Parent(bob, charlie) is True
✅ Applying rule: Parent(x, y) ∧ Parent(y, z) → Grandparent(x, z)
✅ Therefore: Grandparent(alice, charlie)
✅ Added to KB: Grandparent(alice, charlie)


## 💬 Discuss

All three questions are answerable directly from the output of Part 4b above.

1. **The evaluator printed `∃x. Robot(x) -> False`.** The knowledge base has simply never heard of a robot. Is `False` the right answer? Name one system where treating "not recorded" as "not true" is exactly right, and one where it would be dangerous — and say what you would change in the `KnowledgeBase` class to make the difference visible to the caller.
2. **It also printed `∀x. Human(x) -> True`, over a domain of exactly two objects: socrates and plato.** What has that actually established? If a colleague reports "our reasoner confirmed the rule holds universally", what question should you ask first? How does this compare to a machine-learning model reporting 100% accuracy on a nine-row test set?
3. **Japan's Fifth Generation Computer Systems project (MITI, 1982–1994) spent just under ¥57 billion — about US$320 million — building machines whose native operation was logical inference.** The hardware worked and hit its target inference rates; the applications never left demonstration form. In 2026, for which tasks would you still choose hand-written logic over a learned model, and for which the reverse? Pick one task on each side and defend it — "explainability" alone is not an argument until you say who needs the explanation and what they will do with it.

---


## ⚠️ Where this breaks

Three of these limits are visible in this notebook's own output. Read them there first.

- **The closed-world assumption is doing silent work.** `∃x. Robot(x)` evaluated to `False` — not because there are no robots but because none is recorded. The code comment in Part 4 says so outright. Under a closed-world assumption "unknown" and "false" are the same value, which is correct for a parts catalogue and wrong for a patient's allergy list.
- **∀ ranges over the domain you happen to have.** `domain_of(kb)` collects the constants mentioned in the stored facts — here, two of them. `∀x. Human(x) -> True` means "both objects I know about are human", which is not a claim about humanity. Universal quantification over a finite, incidental domain is a much weaker statement than it looks, and this is the single most common way logical claims get oversold.
- **This evaluator handles one quantifier over one predicate.** No nesting (`∀x.∃y.`), no connectives inside the quantifier, no functions, no unification. The summary says as much. Real reasoners add all of that, and each addition costs.
- **Full first-order logic is only *semi*-decidable.** Church and Turing showed in 1936 that no algorithm decides validity for every first-order formula: a prover will eventually confirm a statement that *is* provable, but on one that is not it may simply run forever. That is why practical systems restrict the language — Datalog, description logics, the decidable fragments CodeQL and OWL reasoners use — instead of accepting arbitrary formulas.
- **The expensive part is writing the world down, not reasoning over it.** That is the lesson of the Fifth Generation project above, and of Cyc, which has been hand-encoding common-sense axioms since 1984. Inference is cheap; knowledge acquisition is not.
- **The cheaper alternative.** If your facts fit in tables and your questions are queries, use a database: SQL is first-order logic with an optimiser and fifty years of engineering behind it. Move up to Datalog when you need *recursive* rules over those facts (ancestor, reachability, taint flow), and to a full theorem prover only when you genuinely need proof search.

---


## Summary

### Key Concepts Learned:

1. **First-Order Logic (FOL)**
   - Extends propositional logic with predicates, quantifiers, and variables
   - Can express relationships and properties of objects
   - More expressive than propositional logic

2. **FOL Components**
   - **Predicates**: Human(x), Loves(x, y)
   - **Quantifiers**: Universal (∀) and Existential (∃)
   - **Variables**: x, y, z represent objects
   - **Functions**: Operations on objects

3. **FOL Syntax**
   - Universal: ∀x. P(x) - "For all x, P(x)"
   - Existential: ∃x. P(x) - "There exists x such that P(x)"

4. **FOL Evaluation**
   - Requires a knowledge base (set of facts) and a finite domain of objects
   - ∀ / ∃ formulas are checked by substituting every domain object (Part 4b)
   - Our parser/evaluator handles the simplest shape — one quantifier over one predicate — enough to see the mechanics; full FOL reasoners handle nested quantifiers, connectives, and unification

### Real-World Applications:
- Expert systems (medical diagnosis, legal reasoning)
- Knowledge representation and reasoning
- Database query languages (SQL uses FOL concepts)
- Automated theorem proving
- Natural language understanding

### Difference from Propositional Logic:
- **Propositional Logic**: "It is raining" (simple True/False)
- **FOL**: "For all humans x, x is mortal" (can express general rules)

**Reference:** This notebook covers Course 02, Unit 2 requirements: "Working with First-Order Logic (FOL) syntax and semantics" and "Writing code to parse and evaluate FOL formulas"


## 📚 References

1. Frege, G. (1879). *Begriffsschrift: A Formula Language, Modeled upon that of Arithmetic, for Pure Thought*. Halle. (the origin of quantifiers and predicate logic)
2. Badreddine, S., d'Avila Garcez, A., Serafini, L., & Spranger, M. (2022). *Logic Tensor Networks*. Artificial Intelligence 303, 103649. <https://arxiv.org/abs/2012.13635> (a differentiable first-order logic for modern AI)
3. Russell, S., & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach* (4th ed.), Chapter 8 (First-Order Logic). Pearson. <https://aima.cs.berkeley.edu/>